# Gold Layer — Food Inspections
**Reads from:** `foodlens_silver`  
**Writes to:** `foodlens_gold`


In [0]:
# Parameters
dbutils.widgets.text('catalog', 'workspace')
dbutils.widgets.text('silver',  'foodlens_silver')
dbutils.widgets.text('gold',    'foodlens_gold')
CATALOG = dbutils.widgets.get('catalog')
SILVER  = dbutils.widgets.get('silver')
GOLD    = dbutils.widgets.get('gold')
print(f'Catalog: {CATALOG} | Silver: {SILVER} | Gold: {GOLD}')


In [0]:
# Imports
from pyspark.sql.functions import (
    col, lit, trim, upper, when, to_date, date_format,
    dayofmonth, month, quarter, year, sha2, concat_ws,
    coalesce, current_timestamp, current_date,
    regexp_extract, row_number
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime
from pyspark.sql import Row
print('Imports OK')


In [0]:
# Load Silver Tables
chi_insp = spark.table(f'{CATALOG}.{SILVER}.chicago_inspections')
dal_insp = spark.table(f'{CATALOG}.{SILVER}.dallas_inspections')
chi_viol = spark.table(f'{CATALOG}.{SILVER}.chicago_violations')
dal_viol = spark.table(f'{CATALOG}.{SILVER}.dallas_violations')
print(f'chicago_inspections : {chi_insp.count():,} rows')
print(f'dallas_inspections  : {dal_insp.count():,} rows')
print(f'chicago_violations  : {chi_viol.count():,} rows')
print(f'dallas_violations   : {dal_viol.count():,} rows')


## dim_date


In [0]:
# dim_date
start = datetime.date(2000, 1, 1)
end   = datetime.date(2031, 12, 31)
dates = [Row(full_date=str(start + datetime.timedelta(days=x)))
         for x in range((end - start).days + 1)]

dim_date = spark.createDataFrame(dates) \
    .withColumn('full_date',   to_date('full_date')) \
    .withColumn('day',         dayofmonth('full_date')) \
    .withColumn('month',       month('full_date')) \
    .withColumn('month_name',  date_format('full_date', 'MMMM')) \
    .withColumn('quarter',     quarter('full_date')) \
    .withColumn('year',        year('full_date')) \
    .withColumn('day_of_week', date_format('full_date', 'EEEE')) \
    .withColumn('date_key',    row_number().over(Window.orderBy('full_date')))

dim_date = dim_date.select('date_key','full_date','day','month',
                            'month_name','quarter','year','day_of_week')

dim_date.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_date')
print(f'dim_date: {dim_date.count():,} rows')
dim_date.show(5)


## dim_location


In [0]:
# dim_location
chi_loc = chi_insp.select(
    trim(upper(col('Address'))).alias('address'),
    col('City').alias('city'),
    col('State').alias('state'),
    col('Zip').cast('string').alias('zip_code'),
    col('Latitude').alias('latitude'),
    col('Longitude').alias('longitude')
)

dal_loc = dal_insp.select(
    trim(upper(col('Street Address'))).alias('address'),
    lit('DALLAS').alias('city'),
    lit('TX').alias('state'),
    col('Zip Code').cast('string').alias('zip_code'),
    regexp_extract(col('Lat Long Location'), r'(-?\d+\.\d+),\s*(-?\d+\.\d+)', 1)
        .cast('double').alias('latitude'),
    regexp_extract(col('Lat Long Location'), r'(-?\d+\.\d+),\s*(-?\d+\.\d+)', 2)
        .cast('double').alias('longitude')
)

dim_location = chi_loc.union(dal_loc) \
    .filter(col('address').isNotNull()) \
    .distinct() \
    .withColumn('location_key',
        row_number().over(Window.orderBy('zip_code','address')))

dim_location = dim_location.select(
    'location_key','address','city','state','zip_code','latitude','longitude')

dim_location.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_location')
print(f'dim_location: {dim_location.count():,} rows')
dim_location.show(5)


## dim_inspection_type


In [0]:
# dim_inspection_type
chi_types = chi_insp.select(
    trim(col('Inspection Type')).alias('inspection_type_name'),
    lit('Chicago').alias('city_source')
).distinct()

dal_types = dal_insp.select(
    trim(col('Inspection Type')).alias('inspection_type_name'),
    lit('Dallas').alias('city_source')
).distinct()

dim_inspection_type = chi_types.union(dal_types) \
    .filter(col('inspection_type_name').isNotNull()) \
    .distinct() \
    .withColumn('inspection_type_key',
        row_number().over(Window.orderBy('city_source','inspection_type_name')))

dim_inspection_type = dim_inspection_type.select(
    'inspection_type_key','inspection_type_name','city_source')

dim_inspection_type.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_inspection_type')
print(f'dim_inspection_type: {dim_inspection_type.count()} rows')
dim_inspection_type.show(truncate=False)


## dim_result


In [0]:
# dim_result
chi_results = chi_insp.select(
    trim(col('Results')).alias('result_description'),
    col('inspection_score').alias('derived_score'),
    lit('Chicago').alias('city_source')
).distinct()

dal_results = dal_insp.select(
    lit('NUMERIC_SCORE').alias('result_description'),
    col('Inspection Score').alias('derived_score'),
    lit('Dallas').alias('city_source')
).distinct()

dim_result = chi_results.union(dal_results) \
    .filter(col('result_description').isNotNull()) \
    .distinct() \
    .withColumn('result_key',
        row_number().over(Window.orderBy('city_source','result_description')))

dim_result = dim_result.select(
    'result_key','result_description','derived_score','city_source')

dim_result.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_result')
print(f'dim_result: {dim_result.count()} rows')
dim_result.show(truncate=False)


## dim_risk_category


In [0]:
# dim_risk_category  (Chicago only)
dim_risk_category = chi_insp.select(
    trim(col('Risk')).alias('risk_description'),
    lit('Chicago').alias('city_source')
) \
    .filter(col('risk_description').isNotNull()) \
    .distinct() \
    .withColumn('risk_key',
        row_number().over(Window.orderBy('risk_description')))

dim_risk_category = dim_risk_category.select(
    'risk_key','risk_description','city_source')

dim_risk_category.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_risk_category')
print(f'dim_risk_category: {dim_risk_category.count()} rows')
dim_risk_category.show(truncate=False)


## dim_violation — Standardized schema (both cities)


In [0]:
# dim_violation
chi_violations = chi_viol.select(
    col('violation_code'),
    trim(upper(col('violation_description'))).alias('violation_description'),
    lit('Chicago').alias('city_source')
).distinct()

dal_violations = dal_viol.select(
    lit(None).cast('string').alias('violation_code'),
    trim(upper(col('violation_description'))).alias('violation_description'),
    lit('Dallas').alias('city_source')
).distinct()

dim_violation = chi_violations.union(dal_violations) \
    .filter(col('violation_description').isNotNull()) \
    .distinct() \
    .withColumn('is_critical',
        upper(col('violation_description')).rlike('CRITICAL|URGENT')) \
    .withColumn('violation_key',
        row_number().over(Window.orderBy(
            'city_source','violation_code','violation_description')))

dim_violation = dim_violation.select(
    'violation_key','violation_code',
    'violation_description','city_source','is_critical')

dim_violation.write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.dim_violation')
print(f'dim_violation: {dim_violation.count():,} rows')
dim_violation.show(5, truncate=80)


## dim_establishment — SCD Type 2


###### SCD2 — Manual Implementation using Delta Lake Merge
##### On first run: initial load - all records active
###### On subsequent runs: expire changed records, insert new versions
###### Change detection via SHA2 row_hash on tracked columns

In [0]:
# dim_establishment SCD2
chi_estab = chi_insp.select(
    col("License #").cast("string").alias("establishment_id"),
    trim(col('DBA Name')).alias('dba_name'),
    trim(col('AKA Name')).alias('aka_name'),
    col('License #').cast('string').alias('license_number'),
    trim(col('Facility Type')).alias('facility_type'),
    trim(col('Risk')).alias('risk_category'),
    lit('Chicago').alias('city_source')
).distinct()

dal_estab = dal_insp.select(
    trim(col('Restaurant Name')).alias('establishment_id'),
    trim(col('Restaurant Name')).alias('dba_name'),
    lit(None).cast('string').alias('aka_name'),
    lit(None).cast('string').alias('license_number'),
    lit(None).cast('string').alias('facility_type'),
    lit(None).cast('string').alias('risk_category'),
    lit('Dallas').alias('city_source')
).distinct()

incoming = chi_estab.union(dal_estab) \
    .filter(col('dba_name').isNotNull()) \
    .withColumn('row_hash', sha2(
        concat_ws('||',
            coalesce(col('dba_name'),       lit('')),
            coalesce(col('license_number'), lit('')),
            coalesce(col('facility_type'),  lit('')),
            coalesce(col('risk_category'),  lit(''))
        ), 256))

if not spark.catalog.tableExists(f'{CATALOG}.{GOLD}.dim_establishment'):
    incoming \
        .withColumn('establishment_key',
            row_number().over(Window.orderBy('city_source','establishment_id'))) \
        .withColumn('eff_start_date', lit('2000-01-01').cast('date')) \
        .withColumn('eff_end_date',   lit('9999-12-31').cast('date')) \
        .withColumn('is_current',     lit('Y')) \
        .select('establishment_key','establishment_id','dba_name','aka_name',
                'license_number','facility_type','risk_category','city_source',
                'eff_start_date','eff_end_date','is_current','row_hash') \
        .write.format('delta').mode('overwrite') \
        .option('overwriteSchema','true') \
        .saveAsTable(f'{CATALOG}.{GOLD}.dim_establishment')
    print('dim_establishment: initial load complete')
else:
    dim_table = DeltaTable.forName(spark, f'{CATALOG}.{GOLD}.dim_establishment')
    dim_table.alias('existing').merge(
        incoming.alias('new'),
        'existing.establishment_id = new.establishment_id '
        'AND existing.city_source = new.city_source '
        'AND existing.is_current = \'Y\' '
        'AND existing.row_hash != new.row_hash'
    ).whenMatchedUpdate(set={
        'eff_end_date': current_date(),
        'is_current':   lit('N')
    }).execute()
    incoming \
        .withColumn('establishment_key',
            row_number().over(Window.orderBy('city_source','establishment_id'))) \
        .withColumn('eff_start_date', current_date()) \
        .withColumn('eff_end_date',   lit('9999-12-31').cast('date')) \
        .withColumn('is_current',     lit('Y')) \
        .select('establishment_key','establishment_id','dba_name','aka_name',
                'license_number','facility_type','risk_category','city_source',
                'eff_start_date','eff_end_date','is_current','row_hash') \
        .write.format('delta').mode('append') \
        .saveAsTable(f'{CATALOG}.{GOLD}.dim_establishment')
    print('dim_establishment: SCD2 merge complete')

total = spark.table(f'{CATALOG}.{GOLD}.dim_establishment').count()
print(f'dim_establishment: {total:,} rows total')
spark.table(f'{CATALOG}.{GOLD}.dim_establishment').show(5, truncate=False)


## fact_inspection


In [0]:
# fact_inspection
dim_estab_curr = spark.table(f'{CATALOG}.{GOLD}.dim_establishment').filter(col('is_current')=='Y')
dim_date_tbl   = spark.table(f'{CATALOG}.{GOLD}.dim_date')
dim_loc_tbl    = spark.table(f'{CATALOG}.{GOLD}.dim_location')
dim_type_tbl   = spark.table(f'{CATALOG}.{GOLD}.dim_inspection_type')
dim_res_tbl    = spark.table(f'{CATALOG}.{GOLD}.dim_result')
dim_risk_tbl   = spark.table(f'{CATALOG}.{GOLD}.dim_risk_category')

chi_fact = chi_insp \
    .join(dim_estab_curr.filter(col('city_source')=='Chicago'),
          trim(col('DBA Name')) == dim_estab_curr['dba_name'], 'left') \
    .join(dim_date_tbl,
      col("Inspection Date") == col("full_date"),'left') \
    .join(dim_loc_tbl,
          (trim(upper(chi_insp['Address'])) == dim_loc_tbl['address']) &
          (col('Zip').cast('string') == dim_loc_tbl['zip_code']),'left') \
    .join(dim_type_tbl.filter(col('city_source')=='Chicago'),
          trim(col('Inspection Type')) == dim_type_tbl['inspection_type_name'],'left') \
    .join(dim_res_tbl.filter(col('city_source')=='Chicago'),
          trim(col('Results')) == dim_res_tbl['result_description'],'left') \
    .join(dim_risk_tbl,
          trim(col('Risk')) == dim_risk_tbl['risk_description'],'left') \
    .select(
        col('Inspection ID').cast('string').alias('inspection_id'),
        col('establishment_key'), col('location_key'), col('date_key'),
        col('inspection_type_key'), col('result_key'), col('risk_key'),
        col('License #').cast('string').alias('license_number'),
        col('inspection_score').alias('violation_score'),
        col('violation_count').cast('int'),
        lit('Chicago').alias('city_source')
    )

dal_fact = dal_insp \
    .join(dim_estab_curr.filter(col('city_source')=='Dallas'),
          trim(col('Restaurant Name')) == dim_estab_curr['dba_name'],'left') \
    .join(dim_date_tbl,
      col("Inspection Date") == col("full_date"),'left') \
    .join(dim_loc_tbl,
          (trim(upper(col('Street Address'))) == dim_loc_tbl['address']) &
          (col('Zip Code').cast('string') == dim_loc_tbl['zip_code']),'left') \
    .join(dim_type_tbl.filter(col('city_source')=='Dallas'),
          trim(col('Inspection Type')) == dim_type_tbl['inspection_type_name'],'left') \
    .join(dim_res_tbl.filter(col('city_source')=='Dallas'),
          col('Inspection Score') == dim_res_tbl['derived_score'],'left') \
    .select(
        trim(col('Restaurant Name')).alias('inspection_id'),
        col('establishment_key'), col('location_key'), col('date_key'),
        col('inspection_type_key'), col('result_key'),
        lit(None).cast('int').alias('risk_key'),
        lit(None).cast('string').alias('license_number'),
        col('Inspection Score').alias('violation_score'),
        col('violation_count').cast('int'),
        lit('Dallas').alias('city_source')
    )

chi_fact.union(dal_fact) \
    .withColumn('inspection_key',
        row_number().over(Window.orderBy('city_source','inspection_id'))) \
    .withColumn('record_inserted_at', current_timestamp()) \
    .select('inspection_key','inspection_id','establishment_key','location_key',
            'date_key','inspection_type_key','result_key','risk_key',
            'license_number','violation_score','violation_count',
            'city_source','record_inserted_at') \
    .write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.fact_inspection')

print(f'fact_inspection: {spark.table(f"{CATALOG}.{GOLD}.fact_inspection").count():,} rows')
spark.table(f'{CATALOG}.{GOLD}.fact_inspection').show(5, truncate=False)


## fact_inspection_violation


In [0]:
# fact_inspection_violation
fact_insp_tbl = spark.table(f'{CATALOG}.{GOLD}.fact_inspection')
dim_viol_tbl  = spark.table(f'{CATALOG}.{GOLD}.dim_violation')

chi_bridge = chi_viol \
    .join(fact_insp_tbl.filter(col('city_source')=='Chicago'),
          col('Inspection ID').cast('string') == fact_insp_tbl['inspection_id'],'inner') \
    .join(dim_viol_tbl.filter(col('city_source')=='Chicago'),
          trim(upper(chi_viol['violation_description'])) == dim_viol_tbl['violation_description'],'left') \
    .select(
        col('inspection_key'), col('violation_key'),
        lit(None).cast('int').alias('violation_points'),
        upper(chi_viol['violation_description']).rlike('CRITICAL|URGENT').alias('is_critical')
    )

dal_bridge = dal_viol \
    .join(fact_insp_tbl.filter(col('city_source')=='Dallas'),
          trim(col('Restaurant Name')) == fact_insp_tbl['inspection_id'],'inner') \
    .join(dim_viol_tbl.filter(col('city_source')=='Dallas'),
          trim(upper(dal_viol['violation_description'])) == dim_viol_tbl['violation_description'],'left') \
    .select(
        col('inspection_key'), col('violation_key'),
        col('violation_points'),
        upper(dal_viol['violation_description']).rlike('CRITICAL|URGENT').alias('is_critical')
    )

chi_bridge.union(dal_bridge) \
    .filter(col("violation_key").isNotNull()) \
    .distinct() \
    .withColumn('inspection_violation_key',
        row_number().over(Window.orderBy('inspection_key','violation_key'))) \
    .select('inspection_violation_key','inspection_key',
            'violation_key','violation_points','is_critical') \
    .write.format('delta').mode('overwrite') \
    .option('overwriteSchema','true') \
    .saveAsTable(f'{CATALOG}.{GOLD}.fact_inspection_violation')

print(f'fact_inspection_violation: {spark.table(f"{CATALOG}.{GOLD}.fact_inspection_violation").count():,} rows')
spark.table(f'{CATALOG}.{GOLD}.fact_inspection_violation').show(5)

## Validation — Row Counts + FK Null Checks


In [0]:
# Cell 13: Validation
print('=' * 60)
print('GOLD LAYER SUMMARY')
print('=' * 60)

for t in ['dim_date','dim_location','dim_inspection_type','dim_result',
           'dim_risk_category','dim_violation','dim_establishment',
           'fact_inspection','fact_inspection_violation']:
    cnt = spark.table(f'{CATALOG}.{GOLD}.{t}').count()
    print(f'  {t:<35} {cnt:>10,} rows')

print('\n-- FK Null Check: fact_inspection --')
fi = spark.table(f'{CATALOG}.{GOLD}.fact_inspection')
for fk in ['establishment_key','location_key','date_key',
            'inspection_type_key','result_key']:
    nulls = fi.filter(col(fk).isNull()).count()
    print(f'  {fk:<30} {"OK" if nulls==0 else f"WARNING: {nulls} nulls"}')

print('\n-- FK Null Check: fact_inspection_violation --')
fiv = spark.table(f'{CATALOG}.{GOLD}.fact_inspection_violation')
for fk in ['inspection_key','violation_key']:
    nulls = fiv.filter(col(fk).isNull()).count()
    print(f'  {fk:<30} {"OK" if nulls==0 else f"WARNING: {nulls} nulls"}')

print('\n-- SCD2 Check: dim_establishment --')
de = spark.table(f'{CATALOG}.{GOLD}.dim_establishment')
print(f'  Current (is_current=Y): {de.filter(col("is_current")=="Y").count():,}')
print(f'  Expired (is_current=N): {de.filter(col("is_current")=="N").count():,}')
